# ASL Image Models: HOG + LinearSVM and MobileNetV2 CNN

Trains **Model A** (HOG + LinearSVM) and **Model C** (MobileNetV2) for the three-paradigm comparison.

**Designed for VS Code connected to a Google Colab runtime.**

### Steps
1. Run the **Setup cell** (cell 2) — it finds your `kaggle.json` automatically, downloads the dataset, and clones the repo onto the Colab runtime.
2. Run all remaining cells top-to-bottom.
3. When done, download the output files from `/content/` back into the repo (see Part 5).

In [ ]:
# ── RUNTIME SETUP — run this cell once per session ──────────────────────────
import json, os, shutil, subprocess
from pathlib import Path

# Looks for kaggle.json in the notebook directory and common fallback locations.
# If found automatically, nothing to fill in. Otherwise paste credentials below.
KAGGLE_JSON_MANUAL = {"username": "", "key": ""}  # fallback only

creds = None
for candidate in [
    Path("kaggle.json"),
    Path("../kaggle.json"),
    Path("notebooks/kaggle.json"),
    Path.home() / ".kaggle" / "kaggle.json",
]:
    if candidate.exists():
        creds = json.loads(candidate.read_text())
        print(f"Loaded credentials from {candidate}")
        break

if creds is None:
    assert KAGGLE_JSON_MANUAL["username"], (
        "kaggle.json not found — paste your credentials into KAGGLE_JSON_MANUAL above"
    )
    creds = KAGGLE_JSON_MANUAL

os.environ["KAGGLE_USERNAME"] = creds["username"]
os.environ["KAGGLE_KEY"]      = creds["key"]
kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
(kaggle_dir / "kaggle.json").write_text(json.dumps(creds))
(kaggle_dir / "kaggle.json").chmod(0o600)
print(f"Kaggle authenticated as: {creds['username']}")

# Download and unzip the ASL Alphabet dataset (~1 GB)
if not Path("/content/asl_alphabet_train").exists():
    subprocess.run(["pip", "install", "kaggle", "-q"], check=True)
    subprocess.run(
        ["kaggle", "datasets", "download", "-d", "grassknoted/asl-alphabet",
         "-p", "/content", "--unzip"],
        check=True,
    )
    print("Dataset ready.")
else:
    print("Dataset already present.")

# Clone the repo to get data files (splits.npz, landmarks.npy, etc.)
if not Path("/content/Sign-Language-Project").exists():
    subprocess.run(
        ["git", "clone", "https://github.com/SamrawitDawit/Sign-Language-Project",
         "/content/Sign-Language-Project"],
        check=True,
    )
    print("Repo cloned.")
else:
    print("Repo already present.")

print("Setup complete.")

In [ ]:
import subprocess, sys
for pkg in ["scikit-image", "tqdm", "seaborn"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

In [ ]:
import json
import time
from pathlib import Path

import cv2
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from PIL import Image
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from skimage.feature import hog
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

print("Imports OK")

# Images land at /content/asl_alphabet_train/ after the kaggle unzip
DATASET_ROOT = "/content"

# Repo was cloned to /content/Sign-Language-Project/ by the setup cell
DATA_DIR = Path("/content/Sign-Language-Project/data")

# Outputs go here; download them back to your machine afterwards
OUT_DIR = Path("/content")
OUT_DIR.mkdir(exist_ok=True)

# HOG parameters
HOG_SIZE            = 64
HOG_ORIENTATIONS    = 9
HOG_PIXELS_PER_CELL = (8, 8)
HOG_CELLS_PER_BLOCK = (2, 2)
HOG_BLOCK_NORM      = "L2-Hys"

print(f"DATA_DIR exists : {DATA_DIR.exists()}")
print(f"OUT_DIR  exists : {OUT_DIR.exists()}")
print(f"Images   exist  : {(Path(DATASET_ROOT) / 'asl_alphabet_train').exists()}")

In [ ]:
# Images land at /content/asl_alphabet_train/ after the kaggle unzip
DATASET_ROOT = "/content"

# Repo was cloned to /content/Sign-Language-Project/
DATA_DIR = Path("/content/Sign-Language-Project/data")

# Outputs go here; download them back to your machine afterwards
OUT_DIR = Path("/content")
OUT_DIR.mkdir(exist_ok=True)

# HOG parameters
HOG_SIZE            = 64
HOG_ORIENTATIONS    = 9
HOG_PIXELS_PER_CELL = (8, 8)
HOG_CELLS_PER_BLOCK = (2, 2)
HOG_BLOCK_NORM      = "L2-Hys"

print(f"DATA_DIR exists : {DATA_DIR.exists()}")
print(f"OUT_DIR  exists : {OUT_DIR.exists()}")
print(f"Images   exist  : {Path(DATASET_ROOT, 'asl_alphabet_train').exists()}")

## Part 2 — Load Data and Recover Split Indices

`splits.npz` stores the actual landmark arrays, not row indices. We recover the original
row indices by normalizing `landmarks.npy` and byte-matching against `X_train/val/test`.
Those indices let us look up the image path for each sample in `landmark_metadata.csv`.

In [ ]:
meta = pd.read_csv(DATA_DIR / "landmark_metadata.csv")

with open(DATA_DIR / "label_to_index.json") as f:
    label_to_index: dict = json.load(f)
index_to_label = {v: k for k, v in label_to_index.items()}
num_classes = len(label_to_index)
class_names = [index_to_label[i] for i in range(num_classes)]

print(f"Metadata rows : {len(meta)}")
print(f"Classes       : {num_classes}")
print(f"Columns       : {list(meta.columns)}")
print(meta.head(3))

In [ ]:
print("Loading landmarks.npy ...", end=" ")
lm_raw = np.load(DATA_DIR / "landmarks.npy")
print(f"shape={lm_raw.shape}")

sp   = np.load(DATA_DIR / "splits.npz")
y_tr = sp["y_train"]
y_vl = sp["y_val"]
y_te = sp["y_test"]
print(f"Split sizes — train:{len(y_tr)}  val:{len(y_vl)}  test:{len(y_te)}")

In [ ]:
# Inline normalization (mirrors preprocessing.py)
def _norm(flat: np.ndarray) -> np.ndarray:
    c = flat.astype(np.float32).reshape(21, 3).copy()
    c -= c[0]
    s = np.max(np.linalg.norm(c[:, :2], axis=1))
    if s > 0:
        c /= s
    return c.reshape(-1)

print("Normalizing landmarks for index recovery (~30 s) ...")
lm_norm = np.stack([_norm(r) for r in lm_raw])

lm_map: dict[bytes, int] = {}
for i, row in enumerate(lm_norm):
    k = row.tobytes()
    if k not in lm_map:
        lm_map[k] = i
print(f"Hash map: {len(lm_map)} unique entries")

def _recover(X_split: np.ndarray, y_split: np.ndarray):
    flat = X_split.reshape(len(X_split), -1)
    idxs, ys, miss = [], [], 0
    for row, y in zip(flat, y_split):
        k = row.astype(np.float32).tobytes()
        if k in lm_map:
            idxs.append(lm_map[k]); ys.append(int(y))
        else:
            miss += 1
    return idxs, np.array(ys, dtype=np.int64), miss

tr_idx, y_tr, tr_miss = _recover(sp["X_train"], sp["y_train"])
vl_idx, y_vl, vl_miss = _recover(sp["X_val"],   sp["y_val"])
te_idx, y_te, te_miss = _recover(sp["X_test"],  sp["y_test"])

match_rate = len(tr_idx) / len(sp["X_train"])
print(f"Match rate: {match_rate:.1%}  (misses — train:{tr_miss} val:{vl_miss} test:{te_miss})")

if match_rate < 0.9:
    print("\nLow match — falling back to fixed-seed stratified split (70/15/15)")
    from sklearn.model_selection import train_test_split
    all_y = meta["label"].map(label_to_index).values
    all_i = np.arange(len(meta))
    tv_i, te_i = train_test_split(all_i, test_size=0.15, stratify=all_y, random_state=42)
    tr_i, vl_i = train_test_split(tv_i, test_size=0.15/0.85, stratify=all_y[tv_i], random_state=42)
    tr_idx = tr_i.tolist(); vl_idx = vl_i.tolist(); te_idx = te_i.tolist()
    y_tr = all_y[tr_i]; y_vl = all_y[vl_i]; y_te = all_y[te_i]
    print(f"Fallback split  train:{len(tr_idx)}  val:{len(vl_idx)}  test:{len(te_idx)}")

In [ ]:
# Path remapper: /kaggle/input/<name>/asl_alphabet_train/... → DATASET_ROOT/asl_alphabet_train/...
def img_path(meta_row_idx: int) -> Path:
    raw = meta.iloc[meta_row_idx]["image_path"]
    parts = Path(raw).parts
    try:
        ki = parts.index("input")
        rel = Path(*parts[ki + 2:])   # skip 'input' and dataset name
    except ValueError:
        rel = Path(*parts[1:])        # strip leading '/'
    return Path(DATASET_ROOT) / rel

print("Checking 5 sample paths:")
ok = 0
for i in tr_idx[:5]:
    p = img_path(i)
    exists = p.exists()
    ok += exists
    print(f"  {'OK' if exists else 'MISSING'} {p}")

if ok == 0:
    print("\n*** All paths missing — check DATASET_ROOT in Part 1 ***")

## Part 3 — HOG + LinearSVM

### 3A — Feature Extraction

Grayscale → resize 64×64 → HOG. Features cached to `hog_features.npz` so re-runs skip extraction.

In [ ]:
def extract_hog(path: Path) -> np.ndarray | None:
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    img = cv2.resize(img, (HOG_SIZE, HOG_SIZE))
    return hog(img, orientations=HOG_ORIENTATIONS,
               pixels_per_cell=HOG_PIXELS_PER_CELL,
               cells_per_block=HOG_CELLS_PER_BLOCK,
               block_norm=HOG_BLOCK_NORM)

# Time estimate on 50 images
t0 = time.time()
sample_feat = None
for i in tr_idx[:50]:
    f = extract_hog(img_path(i))
    if f is not None and sample_feat is None:
        sample_feat = f
ms_each   = (time.time() - t0) / 50 * 1000
total_min = ms_each * len(tr_idx) / 60_000
print(f"HOG dim    : {len(sample_feat) if sample_feat is not None else 'N/A'}")
print(f"Speed      : {ms_each:.1f} ms/image")
print(f"Est. total : {total_min:.1f} min for {len(tr_idx)} training images")

In [ ]:
HOG_CACHE = OUT_DIR / "hog_features.npz"

if HOG_CACHE.exists():
    print(f"Loading cached HOG features ...")
    c = np.load(HOG_CACHE)
    H_tr, H_vl, H_te = c["H_tr"], c["H_vl"], c["H_te"]
    y_tr, y_vl, y_te = c["y_tr"], c["y_vl"], c["y_te"]
else:
    def extract_split(idxs, labels, desc):
        X, Y, skipped = [], [], 0
        for i, y in tqdm(zip(idxs, labels), total=len(idxs), desc=desc):
            f = extract_hog(img_path(i))
            if f is not None:
                X.append(f); Y.append(y)
            else:
                skipped += 1
        if skipped:
            print(f"  Skipped {skipped} in '{desc}'")
        return np.array(X, dtype=np.float32), np.array(Y, dtype=np.int64)

    H_tr, y_tr = extract_split(tr_idx, y_tr, "train")
    H_vl, y_vl = extract_split(vl_idx, y_vl, "val  ")
    H_te, y_te = extract_split(te_idx, y_te, "test ")
    np.savez_compressed(HOG_CACHE, H_tr=H_tr, H_vl=H_vl, H_te=H_te,
                        y_tr=y_tr, y_vl=y_vl, y_te=y_te)
    print(f"Cached → {HOG_CACHE}")

print(f"train:{H_tr.shape}  val:{H_vl.shape}  test:{H_te.shape}")

### 3B — Training

In [ ]:
print("Fitting HOG + LinearSVC ...")
t0 = time.time()
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf",    CalibratedClassifierCV(LinearSVC(C=1.0, max_iter=5000), cv=3)),
])
pipeline.fit(H_tr, y_tr)
elapsed_hog = time.time() - t0
print(f"Done in {elapsed_hog:.1f} s")
print(f"Val accuracy : {accuracy_score(y_vl, pipeline.predict(H_vl)):.4f}")

### 3C — Evaluation

In [ ]:
hog_preds    = pipeline.predict(H_te)
test_acc_hog = accuracy_score(y_te, hog_preds)
val_acc_hog  = accuracy_score(y_vl, pipeline.predict(H_vl))
print(f"Test accuracy: {test_acc_hog:.4f}\n")
print(classification_report(y_te, hog_preds, target_names=class_names))

In [ ]:
cm_hog = confusion_matrix(y_te, hog_preds)
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm_hog, annot=False, cmap="Blues",
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"HOG + LinearSVM — Test Accuracy {test_acc_hog:.2%}")
plt.tight_layout()
plt.savefig(OUT_DIR / "confusion_matrix_hog_svm.png", dpi=120)
plt.show()

In [ ]:
joblib.dump(pipeline, OUT_DIR / "model_hog_svm.joblib")
results_hog = {
    "hog_svm": {
        "test_accuracy":  round(float(test_acc_hog), 4),
        "val_accuracy":   round(float(val_acc_hog), 4),
        "train_time_s":   round(elapsed_hog, 1),
        "hog_dim":        int(H_tr.shape[1]),
        "hog_image_size": HOG_SIZE,
    }
}
with open(OUT_DIR / "results_hog_svm.json", "w") as f:
    json.dump(results_hog, f, indent=2)
print("Saved model_hog_svm.joblib + results_hog_svm.json")
print(json.dumps(results_hog, indent=2))

## Part 4 — MobileNetV2 CNN

Transfer learning from ImageNet-pretrained MobileNetV2.
Full fine-tuning at lr=1e-4, 10 epochs, best checkpoint by val accuracy.

### 4A — Dataset and DataLoaders

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class ASLDataset(Dataset):
    def __init__(self, indices, labels, transform):
        self.indices = indices
        self.labels  = labels
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        img = Image.open(img_path(self.indices[i])).convert("RGB")
        return self.transform(img), int(self.labels[i])


BATCH    = 32
train_ds = ASLDataset(tr_idx, y_tr, train_tf)
val_ds   = ASLDataset(vl_idx, y_vl, eval_tf)
test_ds  = ASLDataset(te_idx, y_te, eval_tf)
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
print(f"Train  : {len(train_ds)}  Val : {len(val_ds)}  Test : {len(test_ds)}")

### 4B — Model and Training

In [ ]:
model = models.mobilenet_v2(weights="DEFAULT")
model.classifier[1] = nn.Linear(1280, num_classes)
model = model.to(device)
print(f"MobileNetV2: {sum(p.numel() for p in model.parameters()):,} parameters")

optimizer    = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn      = nn.CrossEntropyLoss()
CNN_EPOCHS   = 10
best_val_acc = 0.0
t0           = time.time()

for epoch in range(1, CNN_EPOCHS + 1):
    model.train()
    for xb, yb in tqdm(train_dl, desc=f"Epoch {epoch}/{CNN_EPOCHS}", leave=False):
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss_fn(model(xb), yb).backward()
        optimizer.step()

    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for xb, yb in val_dl:
            preds.extend(model(xb.to(device)).argmax(1).cpu().numpy())
            trues.extend(yb.numpy())
    v_acc = accuracy_score(trues, preds)
    print(f"  epoch {epoch:2d} | val_acc {v_acc:.4f}")
    if v_acc > best_val_acc:
        best_val_acc = v_acc
        torch.save({"state_dict": model.state_dict(), "num_classes": num_classes},
                   OUT_DIR / "model_cnn.pt")

elapsed_cnn = time.time() - t0
print(f"\nDone in {elapsed_cnn:.1f} s  |  Best val acc: {best_val_acc:.4f}")

### 4C — Evaluation

In [ ]:
ckpt = torch.load(OUT_DIR / "model_cnn.pt", map_location=device, weights_only=True)
model.load_state_dict(ckpt["state_dict"])
model.eval()

preds, trues = [], []
with torch.no_grad():
    for xb, yb in tqdm(test_dl, desc="Test eval"):
        preds.extend(model(xb.to(device)).argmax(1).cpu().numpy())
        trues.extend(yb.numpy())

test_acc_cnn = accuracy_score(trues, preds)
print(f"CNN Test accuracy: {test_acc_cnn:.4f}\n")
print(classification_report(trues, preds, target_names=class_names))

In [ ]:
cm_cnn = confusion_matrix(trues, preds)
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm_cnn, annot=False, cmap="Blues",
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"MobileNetV2 CNN — Test Accuracy {test_acc_cnn:.2%}")
plt.tight_layout()
plt.savefig(OUT_DIR / "confusion_matrix_cnn.png", dpi=120)
plt.show()

In [ ]:
results_cnn = {
    "cnn": {
        "test_accuracy":     round(float(test_acc_cnn), 4),
        "best_val_accuracy": round(float(best_val_acc), 4),
        "train_time_s":      round(elapsed_cnn, 1),
        "epochs":            CNN_EPOCHS,
        "architecture":      "MobileNetV2 (ImageNet pretrained, full fine-tune)",
        "input_size":        224,
    }
}
with open(OUT_DIR / "results_cnn.json", "w") as f:
    json.dump(results_cnn, f, indent=2)
print("Saved model_cnn.pt + results_cnn.json")
print(json.dumps(results_cnn, indent=2))

RESULT_FILES = {
    "HOG + LinearSVM (classical CV)": OUT_DIR / "results_hog_svm.json",
    "MobileNetV2 (CNN)":              OUT_DIR / "results_cnn.json",
    "GCN (graph deep learning)":      DATA_DIR.parent / "results_gcn.json",
    "MLP (baseline)":                 DATA_DIR.parent / "results_mlp.json",
}

print(f"{'Model':<35} {'Test Acc':>10} {'Train Time':>12}")
print("-" * 60)
for name, path in RESULT_FILES.items():
    if not path.exists():
        print(f"  {name:<33}  (not found)")
        continue
    with open(path) as f:
        data = json.load(f)
    key = next(iter(data))
    acc = data[key].get("test_accuracy", "N/A")
    t_s = data[key].get("train_time_s", "N/A")
    acc_str = f"{acc:.4f}" if isinstance(acc, float) else str(acc)
    t_str   = f"{t_s} s"   if isinstance(t_s, (int, float)) else str(t_s)
    print(f"  {name:<33}  {acc_str:>10}  {t_str:>10}")

In [ ]:
RESULT_FILES = {
    "HOG + LinearSVM (classical CV)": OUT_DIR / "results_hog_svm.json",
    "GCN (graph deep learning)":      DATA_DIR.parent / "results_gcn.json",
    "MobileNetV2 (CNN)":              OUT_DIR / "results_cnn.json",
    "MLP (baseline)": DATA_DIR.parent / "results_mlp.json",
}

print(f"{'Model':<35} {'Test Acc':>10} {'Train Time':>12}")
print("-" * 60)
for name, path in RESULT_FILES.items():
    if not path.exists():
        print(f"  {name:<33}  (not found)")
        continue
    with open(path) as f:
        data = json.load(f)
    key = next(iter(data))
    acc = data[key].get("test_accuracy", "N/A")
    t_s = data[key].get("train_time_s", "N/A")
    print(f"  {name:<33}  {acc:>10.4f}  {str(t_s)+' s':>10}")